# MCP Registry Demo
This demo will showcase the following:
* Add a new `MCPRegistry` type using the `Anthropic MCP Registry` application
* Syncing the registry store with `MCPServer` instances in both managed paths:
  * Self-registered server
  * Pre-registered server

## Prerequisites
* Operator deployed on kubernetes (until isse [#1063](https://github.com/stacklok/toolhive/issues/1063) is fixed)

The following command builds and deploys the operator for you:

In [1]:
# !task operator-deploy-local

These settings may be needed depending on your configuration:

In [2]:
# To allow kind to use podman
# !export KIND_EXPERIMENTAL_PROVIDER=podman
# To connect ko builder to podman engine
# !export DOCKER_HOST=unix:///.../podman/podman-lab-api.sock

## Reconciliation Design
A registry instance is reconciled against instances of `MCPServer` type using a `Metadata-Based Discovery Pattern`.

The `MCPServer` instance may includes annotations to hold the required metadata and labels to link the associated registry.

To guarantee independence between the controllers, and allow having servers deployed without registries and vice versa, we
manage two distinct registration paths.

### Self registration path
#### Use case
The server has been started from an existing catalog having all the required metadata.

#### Solution
The server metadata includes both the required annotations, following the standard schema, to register it as a
server and the labels to connect to the registry instance.

Example metadata in an `MCPServer` instance:
```yaml
  annotations:
    mcp.opendatahub.io/server-detail: |
      {
        "id": "4a097d31-ea39-43aa-9af5-6f19a7676901",
        "name": "io.github.manusa/kubernetes-mcp-server",
        "description": "Model Context Protocol (MCP) server for Kubernetes and OpenShift",
        "repository": {
          "url": "https://github.com/manusa/kubernetes-mcp-server",
          "source": "github",
          "id": "930678258"
        },
        "version_detail": {
          "version": "0.0.1-seed",
          "release_date": "2025-05-16T19:09:03Z",
          "is_latest": true
        },
        "packages": [
          {
            "registry_name": "unknown",
            "name": "manusa/kubernetes-mcp-server",
            "version": ""
          }
        ]
      }
  labels:
    toolhive.stacklok.dev/registry-name: example-registry
    toolhive.stacklok.dev/registry-namespace: registry-namespace
```

The `mcp.opendatahub.io/server-detail` annotation is used as-is to define the registration entry.

**TODO** What `MCPServer` status to change if the server-detail is not matching the schema.

### Pre-registration path
#### Use case
The registry has been populated with server entries from an existing catalog.

#### Solution
The server metadata includes the labels to connect to the registry instance and to link it
to a registered instance.

Example metadata in an `MCPServer` instance:
```yaml
  labels:
    toolhive.stacklok.dev/registry-name: example-registry
    toolhive.stacklok.dev/registry-namespace: registry-namespace
    toolhive.stacklok.dev/registered-server-id: 4a097d31-ea39-43aa-9af5-6f19a7676901
```

The `toolhive.stacklok.dev/registered-server-id` label is used top match the existing registration.

**TODO** What `MCPServer` status to change if the registration is not found in the registry.

## Demo
## Preparation steps
Verify no instances are deployed:

In [3]:
!kubectl get mcpserver,mcpregistry

No resources found in toolhive-system namespace.


Deploy the registry:

In [4]:
!kubectl apply -f ./mcpregistry_example.yaml

mcpregistry.toolhive.stacklok.dev/example-registry created


Verify the registry instance:

In [5]:
!kubectl get mcpserver,mcpregistry

NAME                                                 STATUS   SERVERS   AGE
mcpregistry.toolhive.stacklok.dev/example-registry                      2s


The status section contains the Mongo connection data:

In [8]:
!kubectl get mcpregistry example-registry -oyaml | yq '.status'

conditions:
  - lastTransitionTime: "2025-07-31T10:28:04Z"
    message: Registry synced successfully
    reason: Synced
    status: "True"
    type: Ready
lastSyncTime: "2025-07-31T10:28:04Z"
message: Registry synced successfully
mongodbCollectionName: servers_v2
mongodbServiceURL: mongodb://example-registry-mongodb:27017
phase: Ready


On a separate terminal expose the registry service:

In [9]:
# !kubectl port-forward svc/example-registry-registry 8080:8080

On the same terminal, initialize come env vars:

In [10]:
import os
import subprocess

# Get MongoDB pod name
result = subprocess.run(["kubectl", "get", "pod", "-l", "component=mongodb", "-o", "jsonpath={.items[0].metadata.name}"], 
                       capture_output=True, text=True)
mongodb_pod = result.stdout.strip()
os.environ['MONGODB_POD'] = mongodb_pod
print(f"MONGODB_POD={mongodb_pod}")

MONGODB_POD=example-registry-mongodb-658cb66669-c5nfp


### Cleanup DB
Cleanup the `servers_v2` collection in the `mcp-registry` DB:

In [11]:
!kubectl exec -it ${MONGODB_POD} -- mongo mcp-registry --quiet --eval 'db.servers_v2.deleteMany({})'

{ "acknowledged" : true, "deletedCount" : 0 }


Verify count is 0:

In [12]:
!kubectl exec -it ${MONGODB_POD} -- mongo mcp-registry --eval 'db.servers_v2.find().count()'

MongoDB shell version v5.0.31
connecting to: mongodb://127.0.0.1:27017/mcp-registry?compressors=disabled&gssapiServiceName=mongodb
Implicit session: session { "id" : UUID("ba722b3e-2e43-4f5b-bb80-dd6dca77edc4") }
MongoDB server version: 5.0.31
0


### Populate pre-registration
Goal is to pre-register the `everything` server: we're not using the `/push` endpoint of the MCP Registry because it requires a complex configuration of an OAUth token in GitHub. We just add it using the Mongo client instead.

Copy the JSON snippet to the mongodb pod:

In [13]:
!kubectl cp ./everything.json ${MONGODB_POD}:/tmp/everything.json
!kubectl exec -it ${MONGODB_POD} -- mongo mcp-registry --quiet --eval 'db.servers_v2.insertOne(JSON.parse(cat("/tmp/everything.json")))'

{
	"acknowledged" : true,
	"insertedId" : ObjectId("688b455c53d89e931cffbfe0")
}


Verify count is 1:

In [14]:
!kubectl exec -it ${MONGODB_POD} -- mongo mcp-registry --eval 'db.servers_v2.find().count()'

MongoDB shell version v5.0.31
connecting to: mongodb://127.0.0.1:27017/mcp-registry?compressors=disabled&gssapiServiceName=mongodb
Implicit session: session { "id" : UUID("477aaf8a-eb01-496e-8aa8-cc2b3f134271") }
MongoDB server version: 5.0.31
1


Check the DB content includes the registration:

In [15]:
!kubectl exec -it ${MONGODB_POD} -- mongo mcp-registry --quiet --eval 'db.servers_v2.find().toArray()'

[
	{
		"_id" : ObjectId("688b455c53d89e931cffbfe0"),
		"id" : "d34db6e7-6192-46df-9838-bd9694b428a5",
		"name" : "io.github.modelcontextprotocol/everything",
		"description" : "This MCP server attempts to exercise all the features of the MCP protocol",
		"repository" : {
			"url" : "https://github.com/modelcontextprotocol/servers",
			"source" : "github",
			"id" : "890668799"
		},
		"version_detail" : {
			"version" : "0.0.1-seed",
			"release_date" : "2025-07-30T00:23:30Z",
			"is_latest" : true
		},
		"env_vars" : [ ],
		"packages" : [
			{
				"registry_name" : "docker",
				"name" : "mcp/everything",
				"version" : "latest"
			}
		]
	}
]


Access the registry and get the server by ID:

In [20]:
import json
import subprocess

# Get server ID from everything.json
with open('./everything.json', 'r') as f:
    everything_data = json.load(f)
    everything_server_id = everything_data['id']

os.environ['EVERYTHING_SERVER_ID'] = everything_server_id
print(f"EVERYTHING_SERVER_ID={everything_server_id}")

# Make curl request
!curl -s http://localhost:8080/v0/servers/{everything_server_id} | jq

EVERYTHING_SERVER_ID=d34db6e7-6192-46df-9838-bd9694b428a5
{
  "id": "d34db6e7-6192-46df-9838-bd9694b428a5",
  "name": "io.github.modelcontextprotocol/everything",
  "description": "This MCP server attempts to exercise all the features of the MCP protocol",
  "repository": {
    "url": "https://github.com/modelcontextprotocol/servers",
    "source": "github",
    "id": "890668799"
  },
  "version_detail": {
    "version": "0.0.1-seed",
    "release_date": "2025-07-30T00:23:30Z",
    "is_latest": true
  },
  "packages": [
    {
      "registry_name": "docker",
      "name": "mcp/everything",
      "version": "latest"
    }
  ]
}


### Self-registration use case
Deploy the `fetch` `MCPServer` annootated with `toolhive.stacklok.dev/server-detail`

In [21]:
!kubectl apply -f ./mcpserver_fetch.yaml

mcpserver.toolhive.stacklok.dev/fetch created


Fetch the annotation:

In [22]:
!kubectl get mcpserver fetch -oyaml | yq '.metadata.annotations."toolhive.stacklok.dev/server-detail"'

{
  "id": "c2f4ab61-b279-4cb9-932d-5f60bcd1534a",
  "name": "io.github.zcaceres/fetch-mcp",
  "description": "A flexible HTTP fetching Model Context Protocol server.",
  "repository": {
    "url": "https://github.com/zcaceres/fetch-mcp",
    "source": "github",
    "id": "905004409"
  },
  "version_detail": {
    "version": "0.0.1-seed",
    "release_date": "2025-05-16T19:06:44Z",
    "is_latest": true
  },
  "packages": [
    {
      "registry_name": "pypi",
      "name": "fetch",
      "version": "1.0.0"
    }
  ]
}




<div class="alert alert-success">
<b>✅ Verify collection count</b>: expect to be 2 (the new registration and the pre-registered entry)
</div>

In [23]:
!kubectl exec -it ${MONGODB_POD} -- mongo mcp-registry --eval 'db.servers_v2.find().count()'

MongoDB shell version v5.0.31
connecting to: mongodb://127.0.0.1:27017/mcp-registry?compressors=disabled&gssapiServiceName=mongodb
Implicit session: session { "id" : UUID("7aad3d3f-2005-499d-9845-338adab1efa7") }
MongoDB server version: 5.0.31
2


Verify the DB content:

In [24]:
!kubectl exec -it ${MONGODB_POD} -- mongo mcp-registry --quiet --eval 'db.servers_v2.find().toArray()'

[
	{
		"_id" : ObjectId("688b455c53d89e931cffbfe0"),
		"id" : "d34db6e7-6192-46df-9838-bd9694b428a5",
		"name" : "io.github.modelcontextprotocol/everything",
		"description" : "This MCP server attempts to exercise all the features of the MCP protocol",
		"repository" : {
			"url" : "https://github.com/modelcontextprotocol/servers",
			"source" : "github",
			"id" : "890668799"
		},
		"version_detail" : {
			"version" : "0.0.1-seed",
			"release_date" : "2025-07-30T00:23:30Z",
			"is_latest" : true
		},
		"env_vars" : [ ],
		"packages" : [
			{
				"registry_name" : "docker",
				"name" : "mcp/everything",
				"version" : "latest"
			}
		]
	},
	{
		"_id" : ObjectId("688b45b69ed6f3c2c0f4ea01"),
		"id" : "c2f4ab61-b279-4cb9-932d-5f60bcd1534a",
		"name" : "io.github.zcaceres/fetch-mcp",
		"description" : "A flexible HTTP fetching Model Context Protocol server.",
		"repository" : {
			"url" : "https://github.com/zcaceres/fetch-mcp",
			"source" : "github",
			"id" : "905004409"
		},
		"vers

Access the registry and get the server by ID:

In [25]:
import subprocess
import json

# Get fetch server ID
result = subprocess.run(["kubectl", "get", "mcpserver", "fetch", "-oyaml"], capture_output=True, text=True)
yaml_output = result.stdout

# Use yq to extract the annotation and jq to get the id
result2 = subprocess.run(["yq", ".metadata.annotations.\"toolhive.stacklok.dev/server-detail\""], 
                        input=yaml_output, capture_output=True, text=True)
server_detail = json.loads(result2.stdout)
fetch_server_id = server_detail['id']

print(f"FETCH_SERVER_ID={fetch_server_id}")

# Make curl request
!curl -s http://localhost:8080/v0/servers/{fetch_server_id} | jq

FETCH_SERVER_ID=c2f4ab61-b279-4cb9-932d-5f60bcd1534a
{
  "id": "c2f4ab61-b279-4cb9-932d-5f60bcd1534a",
  "name": "io.github.zcaceres/fetch-mcp",
  "description": "A flexible HTTP fetching Model Context Protocol server.",
  "repository": {
    "url": "https://github.com/zcaceres/fetch-mcp",
    "source": "github",
    "id": "905004409"
  },
  "version_detail": {
    "version": "0.0.1-seed",
    "release_date": "2025-05-16T19:06:44Z",
    "is_latest": true
  },
  "packages": [
    {
      "registry_name": "docker",
      "name": "ghcr.io/stackloklabs/gofetch/server",
      "version": "latest"
    }
  ],
  "remotes": [
    {
      "transport_type": "sse",
      "url": "http://mcp-fetch-proxy.toolhive-system.svc.cluster.local:8080"
    }
  ]
}


<div class="alert alert-success">
<b>✅ Verify the existing instance:</b> expected to have the `remotes` entry.
</div> :

In [26]:
!curl -s http://localhost:8080/v0/servers/{fetch_server_id}|jq '.remotes'

[
  {
    "transport_type": "sse",
    "url": "http://mcp-fetch-proxy.toolhive-system.svc.cluster.local:8080"
  }
]


### Pre-registreation use case
Verify the existing instance has no `remotes`:

In [32]:
!curl -s http://localhost:8080/v0/servers/{everything_server_id}|jq '.remotes'

Deploy the `everything` server with pre-registration labels:

In [33]:
!kubectl apply -f ./mcpserver_everything.yaml

mcpserver.toolhive.stacklok.dev/everything created


<div class="alert alert-success">
<b>✅ Verify the existing instance</b>: now it has the `remotes` entry.
</div>:

In [34]:
!curl http://localhost:8080/v0/servers/{everything_server_id}|jq '.remotes'

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
curl: (7) Failed to connect to localhost port 8080 after 0 ms: Couldn't connect to server


## Cleanup
Remove instances:

In [30]:
!kubectl delete mcpserver --all
!kubectl delete mcpregistry --all

mcpserver.toolhive.stacklok.dev "everything" deleted
mcpserver.toolhive.stacklok.dev "fetch" deleted
mcpregistry.toolhive.stacklok.dev "example-registry" deleted


Verify all pods are deleted apart the operator:

In [31]:
!kubectl get pods

NAME                                       READY   STATUS        RESTARTS   AGE
everything-0                               1/1     Terminating   0          3s
example-registry-registry-7f6ff7c7-cq5s9   1/1     Terminating   0          3m4s
toolhive-operator-54889f6448-msm6h         1/1     Running       0          50m
